In [ ]:
import pandas as pd

df = pd.read_csv("btc_1h.csv", index_col=0, parse_dates=True)# ucitavanje podataka
df.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691
2017-08-17 07:00:00,4316.62,4349.99,4287.41,4349.99,4.443249
2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [31]:
#!pip install ta
import ta

#return
df["return_1h"] = df["Close"].pct_change()# koliko se cena promenila u % zato što model lakše uči promene nego apsolutne cene.
df['daily_return'] = df['Close'].pct_change(24)# Dnevni prinos
#mean/std
df["rolling_mean_24h"] = df["Close"].rolling(24).mean()# prosečna cena zadnja 24h
df["rolling_std_24h"] = df["Close"].rolling(24).std()# volatilnost
#lag
df["close_lag_6h"] = df["Close"].shift(6)# cena pre 6h
df["close_lag_12h"] = df["Close"].shift(12)# cena pre 12h
df["close_lag_24h"] = df["Close"].shift(24)# cena pre 24h
df['close_lag_48h'] = df['Close'].shift(48)  # Cena pre 48h
df['close_lag_168h'] = df['Close'].shift(168)  # Cena pre 7 dana

# RSI
df['RSI_14'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()

# MACD
macd = ta.trend.MACD(df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()  # MACD signal linija
df['MACD_hist'] = macd.macd_diff()  # MACD histogram

# Bollinger Bands
bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_mavg'] = bb.bollinger_mavg()  # Srednja Bollinger linija
df['BB_upper'] = bb.bollinger_hband()  # Gornja Bollinger linija
df['BB_lower'] = bb.bollinger_lband()  # Donja Bollinger linija

# SMA i EMA
df['SMA_20'] = ta.trend.SMAIndicator(df['Close'], window=20).sma_indicator()
df['EMA_20'] = ta.trend.EMAIndicator(df['Close'], window=20).ema_indicator()

# ATR (Average True Range)
df['ATR_14'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# ROC (Rate of Change) - Procenat promene cene u poslednjem periodu
df['ROC'] = ta.momentum.ROCIndicator(df['Close'], window=12).roc()

df = df.dropna()# brisanje jer rolling i lag nekada stvaraju prazne vrednosti.
df.head() #ispis :)

,Open,High,Low,Close,Volume,return_1h,daily_return,rolling_mean_24h,rolling_std_24h,close_lag_6h,...,MACD,MACD_signal,MACD_hist,BB_mavg,BB_upper,BB_lower,SMA_20,EMA_20,ATR_14,ROC
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-08-24 04:00:00,4113.58,4148.19,4090.39,4113.98,32.247571,0.000097,0.007440,4145.985833,52.439849,4114.20,...,14.272129,26.070185,-11.798056,4158.5800,4249.760962,4067.399038,4158.5800,4123.466923,73.916662,-2.281456
2017-08-24 05:00:00,4113.98,4177.64,4113.49,4132.09,28.158769,0.004402,0.019469,4149.273750,48.709115,4114.01,...,13.525285,23.561205,-10.035920,4155.0285,4244.510883,4065.546117,4155.0285,4124.288168,73.219043,-0.820398
2017-08-24 06:00:00,4132.09,4177.18,4131.91,4133.42,29.921536,0.000322,0.013093,4151.499583,46.579934,4131.00,...,12.892113,21.427387,-8.535274,4150.2985,4233.637800,4066.959200,4150.2985,4125.157866,71.222683,-0.240143
2017-08-24 07:00:00,4153.97,4173.99,4133.41,4153.32,32.851584,0.004814,0.019250,4154.767917,43.628501,4140.91,...,13.836584,19.909226,-6.072642,4146.0650,4219.123982,4073.006018,4146.0650,4127.839974,69.033920,0.880481
2017-08-24 08:00:00,4153.80,4206.88,4153.32,4200.00,32.275428,0.011239,0.018429,4157.934583,44.054251,4131.92,...,18.142633,19.555907,-1.413274,4145.2810,4215.620914,4074.941086,4145.2810,4134.712358,67.928640,1.981352


In [32]:
#dodaje se future return
df["future_close_24h"] = df["Close"].shift(-24)# uzimamo cenu 24h u budućnosti
df["future_return_24h"] = (df["future_close_24h"] - df["Close"]) / df["Close"]# Model ne predviđa cenu nego promenu

def classify_direction(x):
    if x > 0.02:
        return 2      # UP
    elif x < -0.02:
        return 0      # DOWN
    else:
        return 1      # STABLE

#izbacuje NaN
df["direction_24h"] = df["future_return_24h"].apply(classify_direction)

df = df.dropna()
df["direction_24h"].value_counts(normalize=True)

direction_24h
1    0.592607
2    0.217604
0    0.189789
Name: proportion, dtype: float64

In [33]:
# -----------------------------
# 1) TEMPORAL SPLIT
# -----------------------------


# izdvajamo target
target_col = "direction_24h"

features = [
    "Open","High","Low","Close","Volume",

    "return_1h",
    "daily_return",

    "rolling_mean_24h",
    "rolling_std_24h",

    "close_lag_6h",
    "close_lag_12h",
    "close_lag_24h",
    "close_lag_48h",
    "close_lag_168h",

    "RSI_14",
    "MACD",
    "MACD_signal",
    "MACD_hist",

    "ATR_14",
    "ROC",

    "SMA_20",
    "EMA_20",
    "BB_upper",
    "BB_lower"
]

# indeks za split (60/20/20)
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

#najstariji podaci idu za train, onda za vel i na kraju test
train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print(len(train_df), len(val_df), len(test_df))

44718 14906 14906


In [34]:
# -----------------------------
# 2) SCALING
# -----------------------------

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# FIT samo na train
train_df[features] = scaler.fit_transform(train_df[features])

# TRANSFORM na val i test
val_df[features] = scaler.transform(val_df[features])
test_df[features] = scaler.transform(test_df[features])

In [41]:
train_df.head()

,Open,High,Low,Close,Volume,return_1h,daily_return,rolling_mean_24h,rolling_std_24h,close_lag_6h,...,BB_mavg,BB_upper,BB_lower,SMA_20,EMA_20,ATR_14,ROC,future_close_24h,future_return_24h,direction_24h
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-08-24 04:00:00,-0.902189,-0.901733,-0.901852,-0.902193,-0.805236,0.002096,0.135813,-0.900259,-0.646461,-0.902006,...,4158.5800,-0.900307,-0.898127,-0.899536,-0.901698,-0.650914,-0.815440,4310.20,0.047696,2
2017-08-24 05:00:00,-0.902165,-0.900023,-0.900494,-0.901135,-0.806559,0.470197,0.422273,-0.900067,-0.657659,-0.902017,...,4155.0285,-0.900606,-0.898238,-0.899743,-0.901650,-0.653708,-0.311905,4281.04,0.036047,2
2017-08-24 06:00:00,-0.901107,-0.900050,-0.899412,-0.901057,-0.805988,0.026522,0.270448,-0.899937,-0.664050,-0.901025,...,4150.2985,-0.901226,-0.898154,-0.900020,-0.901599,-0.661703,-0.111927,4302.72,0.040959,2
2017-08-24 07:00:00,-0.899829,-0.900235,-0.899324,-0.899895,-0.805040,0.515036,0.417058,-0.899746,-0.672909,-0.900446,...,4146.0650,-0.902054,-0.897792,-0.900267,-0.901442,-0.670469,0.274282,4329.00,0.042299,2
2017-08-24 08:00:00,-0.899839,-0.898325,-0.898154,-0.897169,-0.805227,1.213660,0.397508,-0.899561,-0.671631,-0.900971,...,4145.2810,-0.902254,-0.897676,-0.900313,-0.901041,-0.674896,0.653683,4332.17,0.031469,2


In [ ]:
# -----------------------------
# 3) Priprema podataka
# -----------------------------

# Definišemo ulazne podatke (features) i ciljne varijable (target)
X_train = train_df[features].values
y_train = train_df[target_col].values

X_val = val_df[features].values
y_val = val_df[target_col].values

X_test = test_df[features].values
y_test = test_df[target_col].values

In [ ]:
# -----------------------------
# 4) Korišćenje Random Forest modela
# -----------------------------

from sklearn.ensemble import RandomForestClassifier


# Inicijalizacija Random Forest modela
#class_weights = {0: 1, 1: 3, 2: 2}# sa ovim je jos gore :'(  (Accuracy on Test Data: 0.21238759898000267)
#rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight=class_weights)
rf = RandomForestClassifier(n_estimators=300, random_state=42) # Bez balansiranja

# Treniranje modela sa trening podacima
rf.fit(X_train, y_train)

# Predikcija na test podacima
y_pred = rf.predict(X_test)


In [ ]:
# -----------------------------
# 5) Evaluacija modela
# -----------------------------

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import numpy as np


# Prikazujemo tačnost modela
print(f"Accuracy on Test Data: {accuracy_score(y_test, y_pred)}")

# Detaljniji izvještaj
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

acc = accuracy_score(y_test, y_pred)
directional_accuracy = np.mean(y_pred == y_test)
f1 = f1_score(y_test, y_pred, average='weighted')  
print(f"Accuracy: {acc:.4f} | Directional Accuracy: {directional_accuracy:.4f} | F1-Score: {f1:.4f}")

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


# Prikazujemo važnost karakteristika
print("\nFeature Importance:")
importances = rf.feature_importances_
for feature, importance in zip(features, importances):
    print(f"{feature}: {importance}")

Accuracy on Test Data: 0.32047497651952234

Classification Report:
              precision    recall  f1-score   support

           0       0.15      0.59      0.24      2366
           1       0.62      0.32      0.43     10126
           2       0.27      0.04      0.07      2414

    accuracy                           0.32     14906
   macro avg       0.35      0.32      0.25     14906
weighted avg       0.49      0.32      0.34     14906

Accuracy: 0.3205 | Directional Accuracy: 0.3205 | F1-Score: 0.3392

Confusion Matrix:
[[1396  913   57]
 [6636 3282  208]
 [1240 1075   99]]

Feature Importance:
Open: 0.03189864922684194
High: 0.03588351758392084
Low: 0.03509273048333819
Close: 0.03808191952443243
Volume: 0.03706210888317397
return_1h: 0.01970320546543963
daily_return: 0.03516014550861554
rolling_mean_24h: 0.045325682533318756
rolling_std_24h: 0.05100160282083178
close_lag_6h: 0.031142549517664674
close_lag_12h: 0.03186264656972583
close_lag_24h: 0.04135936594589034
close_lag_48

In [39]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
print(f"Baseline Accuracy: {dummy.score(X_test, y_test)}")

Baseline Accuracy: 0.679323762243392


In [40]:
import pandas as pd
print("Distribucija u treningu:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nDistribucija u testu:")
print(pd.Series(y_test).value_counts(normalize=True))

Distribucija u treningu:
1    0.522049
2    0.256049
0    0.221902
Name: proportion, dtype: float64

Distribucija u testu:
1    0.679324
2    0.161948
0    0.158728
Name: proportion, dtype: float64
